In [ ]:
%pip uninstall -y tensorflow keras tf-keras tensorflow-text tensorflow-hub tf-models-official >/dev/null 2>&1

%pip install -q \
  "tensorflow==2.19.0" \
  "tensorflow-text==2.19.0" \
  "tensorflow-hub==0.16.1" \
  "tf-keras==2.19.0" \
  "pandas" \
  "scikit-learn" \
  "numpy"

In [ ]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import json
import math
import numpy as np
import pandas as pd
import tensorflow as tf
import tensorflow_hub as hub
import tensorflow_text as text
import tf_keras
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

print("TF:", tf.__version__)
print("Hub:", hub.__version__)
print("TF Text:", text.__version__)
print("tf_keras:", tf_keras.__version__)
print("tf.keras module:", tf.keras)

TF: 2.19.0
Hub: 0.16.1
TF Text: 2.19.0
tf_keras: 2.19.0
tf.keras module: <module 'tf_keras.api._v2.keras' from '/usr/local/lib/python3.12/dist-packages/tf_keras/api/_v2/keras/__init__.py'>


In [ ]:
from dataclasses import dataclass, asdict
from typing import Tuple

# =========================
# Configuration
# =========================
CSV_PATH = "civic_priority_dataset_6000.csv"  # <- change this
OUTPUT_DIR = "priority_bert_outputs"

TITLE_COL = "Title"
DESCRIPTION_COL = "Description"
CATEGORY_COL = "Category"
LABEL_COL = "PriorityScore"

# Accuracy/speed trade-off. These exact Small BERT handles come from TensorFlow's official BERT tutorial.
# Faster/smaller: L-2_H-128_A-2
# Better quality: L-4_H-512_A-8
# Heavier:        L-6_H-512_A-8
PREPROCESS_HANDLE = "https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3"
ENCODER_HANDLE = "https://tfhub.dev/tensorflow/small_bert/bert_en_uncased_L-4_H-512_A-8/1"

MAX_EPOCHS = 5
BATCH_SIZE = 16
LEARNING_RATE = 3e-5
VALIDATION_SIZE = 0.15
TEST_SIZE = 0.15
RANDOM_STATE = 42
DROPOUT = 0.2

# Export settings
EXPORT_SAVED_MODEL = True
EXPORT_TFLITE = True
ENABLE_DYNAMIC_RANGE_QUANT = True


@dataclass
class ExportConfig:
    preprocess_handle: str
    encoder_handle: str
    max_epochs: int
    batch_size: int
    learning_rate: float
    validation_size: float
    test_size: float
    random_state: int
    dropout: float
    title_col: str
    description_col: str
    category_col: str
    label_col: str


# =========================
# Data helpers
# =========================
def build_structured_text(df: pd.DataFrame) -> pd.Series:
    title = df[TITLE_COL].fillna("").astype(str).str.strip()
    description = df[DESCRIPTION_COL].fillna("").astype(str).str.strip()
    category = df[CATEGORY_COL].fillna("").astype(str).str.strip()
    return (
        "title: " + title +
        " description: " + description +
        " category: " + category
    )


def load_dataframe(csv_path: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    required = {TITLE_COL, DESCRIPTION_COL, CATEGORY_COL, LABEL_COL}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")

    df = df.copy()
    df[LABEL_COL] = pd.to_numeric(df[LABEL_COL], errors="coerce")
    df = df.dropna(subset=[LABEL_COL])
    df[LABEL_COL] = df[LABEL_COL].clip(0.0, 1.0).astype("float32")
    df["model_text"] = build_structured_text(df)
    df = df[df["model_text"].str.len() > 0].reset_index(drop=True)

    if len(df) < 50:
        print("WARNING: dataset is very small. The model may overfit badly.")

    return df


def make_datasets(df: pd.DataFrame) -> Tuple[tf.data.Dataset, tf.data.Dataset, tf.data.Dataset, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    train_df, temp_df = train_test_split(
        df,
        test_size=VALIDATION_SIZE + TEST_SIZE,
        random_state=RANDOM_STATE,
        shuffle=True,
    )

    relative_test_size = TEST_SIZE / (VALIDATION_SIZE + TEST_SIZE)
    val_df, test_df = train_test_split(
        temp_df,
        test_size=relative_test_size,
        random_state=RANDOM_STATE,
        shuffle=True,
    )

    def to_ds(part: pd.DataFrame, training: bool) -> tf.data.Dataset:
        x = part["model_text"].astype(str).tolist()
        y = part[LABEL_COL].astype("float32").values
        ds = tf.data.Dataset.from_tensor_slices((x, y))
        if training:
            ds = ds.shuffle(min(len(part), 4096), seed=RANDOM_STATE, reshuffle_each_iteration=True)
        ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
        return ds

    return (
        to_ds(train_df, True),
        to_ds(val_df, False),
        to_ds(test_df, False),
        train_df,
        val_df,
        test_df,
    )


# =========================
# Metrics
# =========================
def regression_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    y_true = np.asarray(y_true, dtype=np.float32)
    y_pred = np.asarray(y_pred, dtype=np.float32)

    mae = float(np.mean(np.abs(y_true - y_pred)))
    mse = float(np.mean((y_true - y_pred) ** 2))
    rmse = float(math.sqrt(mse))

    ss_res = float(np.sum((y_true - y_pred) ** 2))
    ss_tot = float(np.sum((y_true - np.mean(y_true)) ** 2))
    r2 = float(1.0 - (ss_res / ss_tot)) if ss_tot > 0 else float("nan")

    return {
        "mae": mae,
        "mse": mse,
        "rmse": rmse,
        "r2": r2,
    }


# =========================
# Model
# =========================
def build_model() -> tf.keras.Model:
    text_input = tf.keras.layers.Input(shape=(), dtype=tf.string, name="text")

    # TF Hub preprocessing model turns raw strings into BERT inputs.
    preprocess = hub.KerasLayer(PREPROCESS_HANDLE, name="preprocessing")
    encoder_inputs = preprocess(text_input)

    # Fine-tune the pretrained Small BERT encoder.
    encoder = hub.KerasLayer(
        ENCODER_HANDLE,
        trainable=True,
        name="bert_encoder",
    )
    encoder_outputs = encoder(encoder_inputs)
    pooled_output = encoder_outputs["pooled_output"]

    x = tf.keras.layers.Dropout(DROPOUT, name="dropout")(pooled_output)
    x = tf.keras.layers.Dense(128, activation="relu", name="dense_128")(x)
    x = tf.keras.layers.Dropout(DROPOUT, name="dropout_2")(x)

    # Single continuous score in [0.0, 1.0].
    score = tf.keras.layers.Dense(1, activation="sigmoid", name="priority_score")(x)

    model = tf.keras.Model(inputs=text_input, outputs=score)
    return model


def compile_model(model: tf.keras.Model, train_size: int) -> tf.keras.Model:
    optimizer = tf.keras.optimizers.AdamW(
        learning_rate=LEARNING_RATE,
        weight_decay=0.01,
    )

    model.compile(
        optimizer=optimizer,
        loss=tf.keras.losses.Huber(),
        metrics=[
            tf.keras.metrics.MeanAbsoluteError(name="mae"),
            tf.keras.metrics.MeanSquaredError(name="mse"),
        ],
    )
    return model


# =========================
# Export / inference
# =========================
def save_config(out_dir: str):
    cfg = ExportConfig(
        preprocess_handle=PREPROCESS_HANDLE,
        encoder_handle=ENCODER_HANDLE,
        max_epochs=MAX_EPOCHS,
        batch_size=BATCH_SIZE,
        learning_rate=LEARNING_RATE,
        validation_size=VALIDATION_SIZE,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        dropout=DROPOUT,
        title_col=TITLE_COL,
        description_col=DESCRIPTION_COL,
        category_col=CATEGORY_COL,
        label_col=LABEL_COL,
    )
    with open(os.path.join(out_dir, "training_config.json"), "w", encoding="utf-8") as f:
        json.dump(asdict(cfg), f, indent=2)


def export_saved_model(model: tf.keras.Model, out_dir: str) -> str:
    saved_model_dir = os.path.join(out_dir, "saved_model")
    model.export(saved_model_dir)
    return saved_model_dir


def convert_to_tflite(saved_model_dir: str, out_dir: str) -> str:
    tflite_path = os.path.join(out_dir, "priority_small_bert_regressor.tflite")
    converter = tf.lite.TFLiteConverter.from_saved_model(saved_model_dir)
    converter.target_spec.supported_ops = [
        tf.lite.OpsSet.TFLITE_BUILTINS,
        tf.lite.OpsSet.SELECT_TF_OPS,
    ]
    if ENABLE_DYNAMIC_RANGE_QUANT:
        converter.optimizations = [tf.lite.Optimize.DEFAULT]

    # Helps conversion for some transformer graphs.
    converter._experimental_lower_tensor_list_ops = False

    tflite_model = converter.convert()
    with open(tflite_path, "wb") as f:
        f.write(tflite_model)
    return tflite_path


def tflite_predict_text(tflite_path: str, raw_texts):
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()

    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    # This exported model accepts raw string input if conversion succeeds with preprocessing inside the graph.
    input_index = input_details[0]["index"]
    input_value = np.array(raw_texts, dtype=object)
    interpreter.resize_tensor_input(input_index, input_value.shape)
    interpreter.allocate_tensors()
    interpreter.set_tensor(input_index, input_value)
    interpreter.invoke()

    output = interpreter.get_tensor(output_details[0]["index"])
    return np.clip(output.reshape(-1), 0.0, 1.0)


# =========================
# Main
# =========================
def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    print("Loading dataset...")
    df = load_dataframe(CSV_PATH)
    print(f"Loaded {len(df)} rows")

    train_ds, val_ds, test_ds, train_df, val_df, test_df = make_datasets(df)
    print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

    print("Building model...")
    model = build_model()
    model = compile_model(model, len(train_df))
    model.summary()

    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_mae",
            patience=2,
            mode="min",
            restore_best_weights=True,
        ),
        tf.keras.callbacks.ModelCheckpoint(
            filepath=os.path.join(OUTPUT_DIR, "best_model.keras"),
            monitor="val_mae",
            mode="min",
            save_best_only=True,
        ),
    ]

    print("Training...")
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=MAX_EPOCHS,
        callbacks=callbacks,
    )

    print("Evaluating Keras model...")
    test_pred = model.predict(test_ds).reshape(-1)
    test_pred = np.clip(test_pred, 0.0, 1.0)
    test_true = test_df[LABEL_COL].values.astype("float32")
    metrics = regression_metrics(test_true, test_pred)
    print("Test metrics:")
    print(json.dumps(metrics, indent=2))

    with open(os.path.join(OUTPUT_DIR, "test_metrics.json"), "w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=2)

    # Save a few sample predictions for sanity-checking.
    sample_count = min(10, len(test_df))
    sample_rows = test_df.head(sample_count).copy()
    sample_rows["predicted_priority"] = test_pred[:sample_count]
    sample_rows[[TITLE_COL, DESCRIPTION_COL, CATEGORY_COL, LABEL_COL, "predicted_priority"]].to_csv(
        os.path.join(OUTPUT_DIR, "sample_predictions.csv"),
        index=False,
    )

    save_config(OUTPUT_DIR)

    saved_model_dir = None
    if EXPORT_SAVED_MODEL:
        print("Exporting SavedModel...")
        saved_model_dir = export_saved_model(model, OUTPUT_DIR)
        print(f"SavedModel written to: {saved_model_dir}")

    if EXPORT_TFLITE:
        if not saved_model_dir:
            saved_model_dir = export_saved_model(model, OUTPUT_DIR)

        print("Converting to TFLite...")
        try:
            tflite_path = convert_to_tflite(saved_model_dir, OUTPUT_DIR)
            print(f"TFLite written to: {tflite_path}")

            demo_texts = [
                "title: Open manhole on main road description: Large uncovered manhole in a busy traffic lane causing danger to riders and pedestrians category: Road Safety",
                "title: Park bench paint fading description: Bench paint is peeling in one corner of the park but it is still usable category: Public Property",
            ]
            demo_scores = tflite_predict_text(tflite_path, demo_texts)
            for text_value, score in zip(demo_texts, demo_scores):
                print("-" * 80)
                print(text_value)
                print(f"TFLite score: {float(score):.4f}")

        except Exception as e:
            print("TFLite conversion failed.")
            print("This usually means the preprocessing-inside-model export pulled in ops that your environment/runtime cannot convert cleanly.")
            print("The training part is still valid. Error:")
            print(repr(e))
            print(
                "If this happens, keep the trained SavedModel and switch to a numeric-input export path later "
                "(tokenize outside the model, then export only the encoder + regression head)."
            )

    with open(os.path.join(OUTPUT_DIR, "history.json"), "w", encoding="utf-8") as f:
        json.dump(history.history, f, indent=2)

    print("Done.")


if __name__ == "__main__":
    main()


Loading dataset...
Loaded 6000 rows
Train: 4200 | Val: 900 | Test: 900
Building model...
Model: "model"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 text (InputLayer)           [(None,)]                    0         []                            
                                                                                                  
 preprocessing (KerasLayer)  {'input_type_ids': (None,    0         ['text[0][0]']                
                             128),                                                                
                              'input_mask': (None, 128)                                           
                             , 'input_word_ids': (None,                                           
                              128)}                                                               
     

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [ ]:
import shutil

shutil.make_archive("priority_bert_outputs", 'zip', "priority_bert_outputs")

'/content/priority_bert_outputs.zip'

In [ ]:
from google.colab import files

files.download("priority_bert_outputs.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>